In [14]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"H:\HTOC\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: H:\\HTOC\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [15]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_HOURS = 48  # rolling wall-clock window in UTC (not calendar midnights)
INDICATOR_TYPE_NAMES = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "User Agent","Stripped URL"
]
OWNER_NAMES = [
    'HTOC Org',
    'CISA Federal Feed',
    'CMS_CTI',
    'Crowdstrike Falcon Intelligence',
    'DHS CISCP',
    'Intel471',
    'Mandiant Advantage Threat Intelligence',
    'VA_TIP Data',
]
RESULT_PAGE_SIZE = 500  # keep this smaller; same fields, just paged

# Single cutoff instant for TQL, observed_src filter, and workbook filter
_now_utc = datetime.now(pytz.UTC)
_last_observed_cutoff_dt = _now_utc - timedelta(hours=QUERY_LOOKBACK_HOURS)
start = _last_observed_cutoff_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
LAST_OBSERVED_CUTOFF_TS = pd.Timestamp(_last_observed_cutoff_dt)

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) AND '
    f'lastObserved >= "{start}"'
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while True:
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,falsePositives,threatAssess'
            f'&resultStart={result_start}&resultLimit={result_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += result_limit

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
    observed_src = observed_src[observed_src["lastObserved"] >= LAST_OBSERVED_CUTOFF_TS]
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
    # Filter to keep only records where ownerName is 'HTOC Org'
    observed_src = observed_src[observed_src['ownerName'] == 'HTOC Org'].copy()
    # Keep rows where top-level rating >= 3 OR coalesced threatAssessRating >= 3, and
    # (coalesced TA confidence >= 50 OR top-level confidence >= 50).
    # Coalesce flat vs nested threatAssess columns; keep top-level rating/confidence separate for OR.
    _rating_cols = ("threatAssessRating", "threatAssess.threatAssessRating", "rating")
    _confidence_cols = ("threatAssessConfidence", "threatAssess.threatAssessConfidence")

    def _first_non_null_numeric(df, ordered_cols):
        present = [c for c in ordered_cols if c in df.columns]
        if not present:
            return None
        out = pd.to_numeric(df[present[0]], errors="coerce")
        for c in present[1:]:
            s = pd.to_numeric(df[c], errors="coerce")
            out = out.mask(out.isna(), s)
        return out

    _tar = _first_non_null_numeric(observed_src, _rating_cols)
    _tc = _first_non_null_numeric(observed_src, _confidence_cols)
    if _tar is None or _tc is None:
        raise KeyError(
            f"Could not resolve Threat Assess columns. Tried rating={_rating_cols}, "
            f"confidence={_confidence_cols}. Columns: {list(observed_src.columns)}"
        )
    if "rating" in observed_src.columns:
        _r = pd.to_numeric(observed_src["rating"], errors="coerce")
    else:
        _r = pd.Series(float("nan"), index=observed_src.index, dtype=float)

    if "confidence" in observed_src.columns:
        _c = pd.to_numeric(observed_src["confidence"], errors="coerce")
    else:
        _c = pd.Series(float("nan"), index=observed_src.index, dtype=float)

    _pre_ta = len(observed_src)
    # Use >= 50 so a boundary value of 50.0 is included (strict > 50 dropped those rows).
    _pass_rating_band = (_tar >= 3) | (_r >= 3)
    _pass_confidence_band = (_tc >= 50) | (_c >= 50)
    observed_src = observed_src[_pass_rating_band & _pass_confidence_band].copy()
    display(
        f"Threat assess filter ((rating>=3 OR threatAssessRating>=3), confidence>=50) coalescing {_rating_cols} / {_confidence_cols}: "
        f"{_pre_ta} -> {len(observed_src)} rows."
    )
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)

"Threat assess filter ((rating>=3 OR threatAssessRating>=3), confidence>=50) coalescing ('threatAssessRating', 'threatAssess.threatAssessRating', 'rating') / ('threatAssessConfidence', 'threatAssess.threatAssessConfidence'): 1276 -> 1004 rows."

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,source,hostName,dnsActive,whoisActive,associatedGroups.data,description,url,text,indicator,sources
0,14636698790081522,2026-06-03T05:56:46Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-07-27T15:22:51Z,3.0,94.0,1.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2620:1ec:8fa:0:0:0:0:10,"CMS_CTI, HTOC Org"
2,3377699725283739,2026-07-27T11:49:25Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2026-07-27T15:22:51Z,3.0,100.0,3.0,...,NaN,flintazimuth.top,False,False,"[{'id': 3377699725004881, 'dateAdded': '2026-0...",NaN,NaN,NaN,flintazimuth.top,HTOC Org
3,14636698790023461,2026-05-29T03:45:02Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-07-27T15:19:25Z,3.0,92.0,1.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.176.31.150,"CMS_CTI, HTOC Org"
4,11258999070006137,2026-05-28T17:45:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-07-27T15:19:25Z,3.0,90.0,1.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,151.240.47.210,"CMS_CTI, HTOC Org"
5,22517998141061739,2026-07-01T13:06:01Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-07-27T15:19:24Z,3.0,78.0,3.0,...,NaN,NaN,NaN,NaN,NaN,nih_soar - INC9604450 & INC9604362,NaN,NaN,106.75.216.134,HTOC Org
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2614,5269329,2025-01-27T17:27:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-04-25T17:34:39Z,5.0,91.0,5.0,...,https://localfirstbank.com,NaN,NaN,NaN,"[{'id': 547781, 'dateAdded': '2025-01-27T17:27...",Gootloader related indicators. VTI endriched.,localfirstbank.com,NaN,localfirstbank.com,HTOC Org
2615,5269340,2025-01-27T17:29:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-04-25T17:34:32Z,5.0,91.0,5.0,...,https://pianowithjonny.com,NaN,NaN,NaN,"[{'id': 547781, 'dateAdded': '2025-01-27T17:27...",Gootloader related indicators. VTI endriched.,pianowithjonny.com,NaN,pianowithjonny.com,HTOC Org
2616,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90.0,3.0,...,https://www.shorturl.at/,NaN,NaN,NaN,"[{'id': 455233, 'dateAdded': '2024-09-09T11:22...",ACD R&F processed a malspam campaign with a Ne...,www.shorturl.at/,NaN,www.shorturl.at/,HTOC Org
2617,4303591,2023-03-03T13:53:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-24T23:25:10Z,3.0,72.0,3.0,...,NaN,NaN,NaN,NaN,"[{'id': 148157, 'dateAdded': '2023-03-03T13:52...",NaN,aka.ms/o0ukef,NaN,aka.ms/o0ukef,HTOC Org


In [16]:
observed_src[observed_src['indicator'] == '174.128.251.99']

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,source,hostName,dnsActive,whoisActive,associatedGroups.data,description,url,text,indicator,sources
1330,5629499574089560,2025-10-21T11:33:56Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-07-26T17:18:54Z,5.0,66.0,2.0,...,NaN,NaN,NaN,NaN,"[{'id': 13510798882117954, 'dateAdded': '2026-...",Key Findings\nSilent Push Threat Analysts have...,NaN,NaN,174.128.251.99,HTOC Org


In [17]:
import pandas as pd
import ast
from datetime import datetime, timedelta
import pytz

# Load the Excel file
file_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"
df = pd.read_excel(file_path)


# Keep only indicators that are also in observed_src
_indicator_col = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in df.columns), None)
if _indicator_col is None:
    raise KeyError(f"Could not find indicator column in df. Columns: {list(df.columns)}")

_observed_indicators = set(observed_src["indicator"].dropna().astype(str))
df = df[df[_indicator_col].astype(str).isin(_observed_indicators)].copy()

# Last Observed column: values come only from observed_src (ThreatConnect), not the workbook
_last_observed_col = next(
    (
        c
        for c in [
            "Last Observed",
            "lastObserved",
            "LastObserved",
            "last_observed",
            "LAST OBSERVED",
        ]
        if c in df.columns
    ),
    None,
)
if _last_observed_col is None:
    raise KeyError(f"Could not find 'Last Observed' column in df. Columns: {list(df.columns)}")

_assoc_groups_src_col = "associatedGroups.data"
_assoc_groups_target_col = "Associated Groups"
if _assoc_groups_src_col not in observed_src.columns:
    raise KeyError(
        f"Could not find '{_assoc_groups_src_col}' column in observed_src. Columns: {list(observed_src.columns)}"
    )


def _extract_group_ids(value):
    # Handle scalar nulls safely; avoid pd.isna on list-like values.
    if value is None:
        return pd.NA

    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return pd.NA
        try:
            parsed = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            return text
    elif isinstance(value, float) and pd.isna(value):
        return pd.NA

    if isinstance(parsed, dict):
        gid = parsed.get("id")
        return f"Group Id: {gid}" if gid is not None else pd.NA

    if isinstance(parsed, list):
        ids = []
        for item in parsed:
            if isinstance(item, dict) and item.get("id") is not None:
                ids.append(f"Group Id: {item.get('id')}")
        return ", ".join(ids) if ids else pd.NA

    return pd.NA


_observed_latest = (
    observed_src.dropna(subset=["indicator"])
    .assign(
        indicator=lambda d: d["indicator"].astype(str),
        lastObserved=lambda d: pd.to_datetime(d["lastObserved"], utc=True, errors="coerce"),
    )
    .sort_values("lastObserved")
    .drop_duplicates(subset=["indicator"], keep="last")
)

_last_obs_by_indicator = _observed_latest.set_index("indicator")["lastObserved"]
_assoc_groups_by_indicator = _observed_latest.set_index("indicator")[_assoc_groups_src_col].map(_extract_group_ids)

# Last Observed: only from ThreatConnect (observed_src); do not fall back to Excel dates
_df_ind = df[_indicator_col].astype(str)
df[_last_observed_col] = pd.to_datetime(_df_ind.map(_last_obs_by_indicator), utc=True, errors="coerce")
_qh = QUERY_LOOKBACK_HOURS if "QUERY_LOOKBACK_HOURS" in globals() else 48
_last_obs_cutoff = (
    LAST_OBSERVED_CUTOFF_TS
    if "LAST_OBSERVED_CUTOFF_TS" in globals()
    else pd.Timestamp(datetime.now(pytz.UTC) - timedelta(hours=_qh), tz="UTC")
)
_pre_lo = len(df)
df = df[df[_last_observed_col].notna() & (df[_last_observed_col] >= _last_obs_cutoff)].copy()
display(
    f"Last Observed filter (ThreatConnect only, >= {_last_obs_cutoff}): {_pre_lo} -> {len(df)} rows."
)

_df_ind = df[_indicator_col].astype(str)

# Add associatedGroups.data ids from observed_src by indicator, stored as 'Associated Groups'
if _assoc_groups_target_col in df.columns:
    df[_assoc_groups_target_col] = _df_ind.map(_assoc_groups_by_indicator).combine_first(df[_assoc_groups_target_col])
else:
    df[_assoc_groups_target_col] = _df_ind.map(_assoc_groups_by_indicator)

df

'Last Observed filter (ThreatConnect only, >= 2026-07-25 15:24:39.181745+00:00): 997 -> 997 rows.'

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,Tagging Boost,Tagging Boost Reason,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups
0,102.0.22.10,2026-07-26 00:00:00+00:00,Address,8,3,0.999562,0,0,"DHA, VA",NaN,NaN,0.0,NaN,180,469,176,low,[2026-07-27] Severity: low. VT score: 2. Conte...,<NA>
1,102.222.90.243,2026-07-26 00:00:00+00:00,Address,16,3,0.999123,0,0,"DHA, VA",NaN,NaN,0.0,NaN,180,469,87,low,[2026-07-27] Severity: low. VT score: 0. Conte...,<NA>
2,102.222.90.244,2026-07-26 00:00:00+00:00,Address,20,3,0.998904,0,0,"NIH, VA",NaN,NaN,0.0,NaN,180,469,140,low,[2026-07-27] Severity: low. VT score: 1. Conte...,<NA>
3,102.88.54.138,2026-07-26 00:00:00+00:00,Address,22,3,0.998795,0,0,"DHA, VA",NaN,NaN,0.0,NaN,180,469,175,low,[2026-07-27] Severity: low. VT score: 2. Conte...,<NA>
5,102.88.55.114,2026-07-26 00:00:00+00:00,Address,14,3,0.999233,0,0,"CMS, DHA, HRSA, OS, VA",NaN,NaN,0.0,NaN,180,469,153,low,[2026-07-27] Severity: low. VT score: 1. Conte...,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3280,rmi.org,2026-07-27 00:00:00+00:00,Stripped URL,41,5,0.997753,0,0,NaN,Event:6755399474000400,NaN,0.0,NaN,0,1000,76,low,[2026-05-27] Severity: low. VT score not avail...,Group Id: 6755399474000400
3287,snowbrains.com,2026-07-26 00:00:00+00:00,Stripped URL,2,5,0.999890,0,0,"DHA, NIH",Event:11258999069000227,NaN,0.0,NaN,0,1000,149,low,[2026-05-27] Severity: low. VT score not avail...,Group Id: 11258999069000227
3293,thebigmansworld.com,2026-07-27 00:00:00+00:00,Stripped URL,148,5,0.991890,0,0,"CMS, DHA, IHS, NIH",Event:5629499566001089,NaN,0.0,NaN,0,1000,145,low,[2026-05-27] Severity: low. VT score not avail...,Group Id: 5629499566001089
3301,utmost.org,2026-07-27 00:00:00+00:00,Stripped URL,99,5,0.994575,0,0,"DHA, NIH",Event:6755399471000728,NaN,0.0,NaN,0,1000,135,low,[2026-05-27] Severity: low. VT score not avail...,Group Id: 6755399471000728


In [18]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=2)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_30524\1484081131.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260727.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260726.csv']
Loaded data from 2 files.


In [19]:
observed_data_df[observed_data_df['indicator'] == '174.128.251.99']

,indicator,API_UserName,obs_date,OpDiv,indicator_key,observations,curr_date
4353,174.128.251.99,50189120947314147395,2026-07-26,NIH,174.128.251.99_NIH,3,2026-07-26


In [20]:
# Keep only indicators that are present in observed_data_df and seen by 2+ OpDiv partners
_indicator_col_df = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in df.columns), None)
_indicator_col_obs = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in observed_data_df.columns), None)
_opdiv_col = next((c for c in ["OpDiv", "opdiv", "OPDIV"] if c in observed_data_df.columns), None)

if _indicator_col_df is None:
    raise KeyError(f"Could not find indicator column in df. Columns: {list(df.columns)}")
if _indicator_col_obs is None:
    raise KeyError(f"Could not find indicator column in observed_data_df. Columns: {list(observed_data_df.columns)}")
if _opdiv_col is None:
    raise KeyError(f"Could not find OpDiv column in observed_data_df. Columns: {list(observed_data_df.columns)}")

obs = observed_data_df.dropna(subset=[_indicator_col_obs, _opdiv_col]).copy()
obs[_indicator_col_obs] = obs[_indicator_col_obs].astype(str).str.strip()
obs[_opdiv_col] = obs[_opdiv_col].astype(str).str.strip()

partners_by_indicator = (
    obs.groupby(_indicator_col_obs)[_opdiv_col]
    .apply(lambda s: sorted(set(x for x in s if x)))
)

eligible_partners = partners_by_indicator[partners_by_indicator.str.len() >= 2]
opdiv_map = eligible_partners.apply(lambda vals: ", ".join(vals))

last_24h_multiple_partners = df[
    df[_indicator_col_df].astype(str).str.strip().isin(eligible_partners.index)
].copy()
last_24h_multiple_partners["OpDiv"] = (
    last_24h_multiple_partners[_indicator_col_df].astype(str).str.strip().map(opdiv_map)
)
last_24h_multiple_partners["Partners"] = last_24h_multiple_partners["OpDiv"]

last_24h_multiple_partners

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,Tagging Boost,Tagging Boost Reason,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,OpDiv
1,102.222.90.243,2026-07-26 00:00:00+00:00,Address,16,3,0.999123,0,0,"DHA, VA",NaN,NaN,0.0,NaN,180,469,87,low,[2026-07-27] Severity: low. VT score: 0. Conte...,<NA>,"DHA, VA"
11,103.120.116.162,2026-07-27 00:00:00+00:00,Address,181,3,0.990082,0,0,"CMS, FDA, HHS, HRSA, OS, VA",Incident:INC9385644,NaN,0.0,NaN,950,854,236,medium,[2026-07-27] Severity: medium. VT score: 10. C...,<NA>,"CMS, FDA, HHS, HRSA, OS, VA"
62,103.125.146.7,2026-07-27 00:00:00+00:00,Address,24,3,0.998685,0,0,"OS, VA",NaN,NaN,0.0,NaN,180,368,139,low,[2026-07-27] Severity: low. VT score: 1. Conte...,<NA>,"OS, VA"
94,103.215.75.19,2026-07-27 00:00:00+00:00,Address,10,3,0.999452,0,0,"HRSA, VA",NaN,NaN,0.0,NaN,850,804,261,medium,[2026-07-27] Severity: medium. VT score: 4. Co...,<NA>,"HRSA, VA"
95,103.59.75.106,2026-07-26 00:00:00+00:00,Address,4,3,0.999781,0,0,"CMS, VA",NaN,NaN,0.0,NaN,180,469,140,low,[2026-07-27] Severity: low. VT score: 1. Conte...,<NA>,"CMS, VA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1932,45.126.43.37,2026-07-27 00:00:00+00:00,Address,4,3,0.999781,0,0,"DHA, NIH, VA",NaN,NaN,0.0,NaN,730,744,209,medium,[2026-07-27] Severity: medium. VT score: 2. Co...,<NA>,"DHA, NIH, VA"
1933,45.198.224.5,2026-07-27 00:00:00+00:00,Address,39,3,0.997863,0,0,"HRSA, IHS, NIH, OS, VA",Incident:INC9636521,NaN,0.0,NaN,970,892,350,medium,[2026-07-27] Severity: medium. VT score: 19. C...,<NA>,"HRSA, IHS, NIH, OS, VA"
1938,87.106.48.104,2026-07-27 00:00:00+00:00,Address,2,3,0.999890,0,0,"NIH, VA",NaN,NaN,0.0,NaN,170,464,100,low,[2026-07-27] Severity: low. VT score: 0. Conte...,<NA>,"NIH, VA"
3222,chaturbate.com,2026-07-27 00:00:00+00:00,Stripped URL,364,3,0.980055,0,0,"DHA, NIH",Incident:6755399448002016,NaN,0.0,NaN,0,1000,142,low,[2026-05-27] Severity: low. VT score not avail...,Group Id: 6755399448002016,"DHA, NIH"


In [21]:
# Filter multi-partner, last-24h indicators to VT score >= 10 based on Explanation text
vt_scores = last_24h_multiple_partners['Explanation'].str.extract(r'VT score:\s*(\d+)', expand=False)
vt_scores = pd.to_numeric(vt_scores, errors='coerce')

last_24h_multi_partners_vt15 = last_24h_multiple_partners[vt_scores >= 2]

last_24h_multi_partners_vt15

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,Tagging Boost,Tagging Boost Reason,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,OpDiv
11,103.120.116.162,2026-07-27 00:00:00+00:00,Address,181,3,0.990082,0,0,"CMS, FDA, HHS, HRSA, OS, VA",Incident:INC9385644,NaN,0.0,NaN,950,854,236,medium,[2026-07-27] Severity: medium. VT score: 10. C...,<NA>,"CMS, FDA, HHS, HRSA, OS, VA"
94,103.215.75.19,2026-07-27 00:00:00+00:00,Address,10,3,0.999452,0,0,"HRSA, VA",NaN,NaN,0.0,NaN,850,804,261,medium,[2026-07-27] Severity: medium. VT score: 4. Co...,<NA>,"HRSA, VA"
133,106.15.238.36,2026-07-27 00:00:00+00:00,Address,112,3,0.993863,0,0,"CMS, FDA, HHS, HRSA, IHS, NIH, OS",Incident:INC9480324,NaN,0.0,NaN,690,623,87,low,[2026-07-27] Severity: low. VT score: 8. Conte...,<NA>,"CMS, FDA, HHS, HRSA, IHS, NIH, OS"
135,106.75.216.134,2026-07-27 00:00:00+00:00,Address,27,3,0.998521,0,0,"CMS, HRSA, OS",Incident:INC9604450;Incident:INC9604362,NaN,0.0,NaN,810,905,112,low,[2026-07-27] Severity: low. VT score: 2. Conte...,<NA>,"CMS, HRSA, OS"
140,107.189.19.172,2026-07-27 00:00:00+00:00,Address,57,3,0.996877,0,0,"HRSA, IHS, NIH, OS, VA",NaN,NaN,0.0,NaN,180,469,191,low,[2026-07-27] Severity: low. VT score: 2. Conte...,<NA>,"HRSA, IHS, NIH, OS, VA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1828,92.118.39.71,2026-07-27 00:00:00+00:00,Address,27,3,0.998521,0,0,"HRSA, IHS, NIH, OS, VA",Incident:INC9604450;Incident:INC9604362,NaN,0.0,NaN,930,844,95,low,[2026-07-27] Severity: low. VT score: 10. Cont...,<NA>,"HRSA, IHS, NIH, OS, VA"
1833,93.152.221.13,2026-07-27 00:00:00+00:00,Address,7,3,0.999616,0,0,"HRSA, NIH, OS",NaN,NaN,0.0,NaN,890,824,500,high,[2026-07-27] Severity: high. VT score: 13. Con...,<NA>,"HRSA, NIH, OS"
1898,134.199.171.81,2026-07-26 00:00:00+00:00,Address,1,3,0.999945,0,0,"HRSA, OS, VA",NaN,NaN,0.0,NaN,170,464,282,medium,[2026-07-27] Severity: medium. VT score: 5. Co...,<NA>,"HRSA, OS, VA"
1932,45.126.43.37,2026-07-27 00:00:00+00:00,Address,4,3,0.999781,0,0,"DHA, NIH, VA",NaN,NaN,0.0,NaN,730,744,209,medium,[2026-07-27] Severity: medium. VT score: 2. Co...,<NA>,"DHA, NIH, VA"


In [22]:
# Keep only high or critical indicators from the VT>=10, multi-partner, last-24h set
final_indicators = last_24h_multi_partners_vt15[last_24h_multi_partners_vt15['Severity'].isin(['high', 'critical'])]

final_indicators

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,Tagging Boost,Tagging Boost Reason,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,OpDiv
586,176.123.5.126,2026-07-27 00:00:00+00:00,Address,21,5,0.998849,0,0,"HRSA, IHS, OS",NaN,Lace Tempest,1.0,cve:cve-2024-50623,360,477,620,high,[2026-07-27] Severity: high. VT score: 15. Con...,"Group Id: 5629499537000808, Group Id: 45035996...","HRSA, IHS, OS"
669,185.220.101.46,2026-07-27 00:00:00+00:00,Address,33,3,0.998192,0,0,"CMS, HRSA",NaN,NaN,0.0,NaN,190,447,616,high,[2026-07-27] Severity: high. VT score: 15. Con...,"Group Id: 136273, Group Id: 129288","CMS, HRSA"
716,185.93.89.147,2026-07-27 00:00:00+00:00,Address,38,3,0.997918,0,0,"HRSA, NIH, OS",NaN,NaN,0.0,NaN,780,742,526,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 22517998136854987,"HRSA, NIH, OS"
1190,45.84.107.172,2026-07-27 00:00:00+00:00,Address,320,3,0.982466,0,0,"CMS, HRSA, OS",NaN,"Seashell Blizzard, UNC6040, UNC6395",1.0,pair:russia+gru,590,631,578,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 6755399498003155,"CMS, HRSA, OS"
1196,46.151.182.31,2026-07-27 00:00:00+00:00,Address,12,3,0.999342,0,0,"HRSA, IHS, OS, VA",NaN,NaN,1.0,standalone:command and control,180,433,620,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 4503599631000000,"HRSA, IHS, OS, VA"
1733,77.247.126.189,2026-07-27 00:00:00+00:00,Address,94,5,0.994849,0,0,"CMS, FDA, HHS, HRSA, OS",NaN,"Diamond Sleet, Emerald Sleet, Famous Chollima,...",1.0,pair:north korea+diamond sleet,280,492,579,high,[2026-07-27] Severity: high. VT score: 11. Con...,"Group Id: 6755399468000794, Group Id: 56294995...","CMS, FDA, HHS, HRSA, OS"
1833,93.152.221.13,2026-07-27 00:00:00+00:00,Address,7,3,0.999616,0,0,"HRSA, NIH, OS",NaN,NaN,0.0,NaN,890,824,500,high,[2026-07-27] Severity: high. VT score: 13. Con...,<NA>,"HRSA, NIH, OS"


In [23]:
import pandas as pd

# Load external tags data
tags_path = r"Z:\HTOC\Data_Analytics\Data\Observed_Tags\htoc_observed_indicator_tags.csv"
tags_df = pd.read_csv(tags_path)

# The indicator column in the tags CSV could be e.g. 'Indicator' or 'indicator'
tags_indicator_col = None
for col in tags_df.columns:
    if str(col).lower() == 'indicator':
        tags_indicator_col = col
        break
if tags_indicator_col is None:
    raise ValueError("Could not find an 'Indicator' column in the tags CSV.")

# The tags field might be called 'Tags', 'tags', or similar
# The tags field might be called 'Tags', 'tags', 'Tag', 'tag', etc.
tags_value_col = None
for col in tags_df.columns:
    if str(col).lower() in ('tags', 'tag'):
        tags_value_col = col
        break
if tags_value_col is None:
    raise ValueError(
        f"Could not find a 'Tag' or 'Tags' column in the tags CSV. "
        f"Available columns: {list(tags_df.columns)}"
    )
# For fast lookup, set up a mapping of indicator -> tags value.
indicator_to_tags = tags_df.set_index(tags_indicator_col)[tags_value_col].to_dict()

# Prepare 'Tags' values for final_indicators
final_tags = final_indicators['Indicator'].map(indicator_to_tags)

# Insert the 'Tags' column as the second to last column
final_cols = list(final_indicators.columns)
if 'Tags' in final_cols:
    final_cols.remove('Tags')
second_to_last_idx = -1 if len(final_cols) == 0 else -1
new_cols = final_cols[:second_to_last_idx] + ['Tags'] + final_cols[second_to_last_idx:]

final_indicators['Tags'] = final_tags
final_indicators = final_indicators[new_cols]
final_indicators


C:\Users\jaskew\AppData\Local\Temp\ipykernel_30524\3834435136.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_indicators['Tags'] = final_tags


,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,...,Tagging Boost,Tagging Boost Reason,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,Tags,OpDiv
586,176.123.5.126,2026-07-27 00:00:00+00:00,Address,21,5,0.998849,0,0,"HRSA, IHS, OS",NaN,...,1.0,cve:cve-2024-50623,360,477,620,high,[2026-07-27] Severity: high. VT score: 15. Con...,"Group Id: 5629499537000808, Group Id: 45035996...",Lace Tempest,"HRSA, IHS, OS"
669,185.220.101.46,2026-07-27 00:00:00+00:00,Address,33,3,0.998192,0,0,"CMS, HRSA",NaN,...,0.0,NaN,190,447,616,high,[2026-07-27] Severity: high. VT score: 15. Con...,"Group Id: 136273, Group Id: 129288",ms-isac,"CMS, HRSA"
716,185.93.89.147,2026-07-27 00:00:00+00:00,Address,38,3,0.997918,0,0,"HRSA, NIH, OS",NaN,...,0.0,NaN,780,742,526,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 22517998136854987,SystemBC,"HRSA, NIH, OS"
1190,45.84.107.172,2026-07-27 00:00:00+00:00,Address,320,3,0.982466,0,0,"CMS, HRSA, OS",NaN,...,1.0,pair:russia+gru,590,631,578,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 6755399498003155,Russia,"CMS, HRSA, OS"
1196,46.151.182.31,2026-07-27 00:00:00+00:00,Address,12,3,0.999342,0,0,"HRSA, IHS, OS, VA",NaN,...,1.0,standalone:command and control,180,433,620,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 4503599631000000,Command and Control,"HRSA, IHS, OS, VA"
1733,77.247.126.189,2026-07-27 00:00:00+00:00,Address,94,5,0.994849,0,0,"CMS, FDA, HHS, HRSA, OS",NaN,...,1.0,pair:north korea+diamond sleet,280,492,579,high,[2026-07-27] Severity: high. VT score: 11. Con...,"Group Id: 6755399468000794, Group Id: 56294995...",North Korea,"CMS, FDA, HHS, HRSA, OS"
1833,93.152.221.13,2026-07-27 00:00:00+00:00,Address,7,3,0.999616,0,0,"HRSA, NIH, OS",NaN,...,0.0,NaN,890,824,500,high,[2026-07-27] Severity: high. VT score: 13. Con...,<NA>,NaN,"HRSA, NIH, OS"


In [24]:
import pandas as pd

# Helper to see if an indicator has an I&W tag
def has_iw(tags_value):
    """
    tags_value is typically a list of dicts from ThreatConnect, e.g.:
    [{'name': 'I&W'}, {'name': 'something else'}, ...]
    """
    if tags_value is None or (isinstance(tags_value, float) and pd.isna(tags_value)):
        return False

    if not isinstance(tags_value, (list, tuple)):
        return False

    for t in tags_value:
        try:
            if isinstance(t, dict):
                name = str(t.get('name', '')).strip()
            else:
                name = str(t).strip()

            if name.lower() in {"i&w", "i & w", "iw"}:
                return True
        except Exception:
            continue
    return False

# 1) Add has_iw flag to observed_src if tags.data exists
if 'tags.data' in observed_src.columns:
    observed_src['has_iw'] = observed_src['tags.data'].apply(has_iw)
else:
    observed_src['has_iw'] = False

# 2) Collapse to one flag per indicator
iw_per_indicator = (
    observed_src.groupby('indicator', dropna=False)['has_iw']
    .max()  # any True -> True
    .reset_index()
    .rename(columns={'indicator': 'Indicator', 'has_iw': 'Reported I&W?_raw'})
)

# 3) Drop ANY existing Reported I&W? variants (_x, _y, etc.)
cols_to_drop = [c for c in final_indicators.columns if c.startswith('Reported I&W?')]
final_indicators = final_indicators.drop(columns=cols_to_drop, errors='ignore')

# 4) Merge once, with a temporary raw boolean column
final_indicators = final_indicators.merge(
    iw_per_indicator,
    on='Indicator',
    how='left'
)

# 5) Convert to Yes/No, defaulting missing to 'No'
final_indicators['Reported I&W?'] = (
    final_indicators['Reported I&W?_raw']
    .fillna(False)
    .map({True: 'Yes', False: 'No'})
)

# 6) Drop the temporary raw column
final_indicators = final_indicators.drop(columns=['Reported I&W?_raw'])

# Rename column 'HTOC Threat Score' to 'PRISM Score' if it exists
if "HTOC Threat Score" in final_indicators.columns:
    final_indicators = final_indicators.rename(columns={"HTOC Threat Score": "PRISM Score"})


final_indicators

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,...,Tagging Boost Reason,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,Tags,OpDiv,Reported I&W?
0,176.123.5.126,2026-07-27 00:00:00+00:00,Address,21,5,0.998849,0,0,"HRSA, IHS, OS",NaN,...,cve:cve-2024-50623,360,477,620,high,[2026-07-27] Severity: high. VT score: 15. Con...,"Group Id: 5629499537000808, Group Id: 45035996...",Lace Tempest,"HRSA, IHS, OS",Yes
1,185.220.101.46,2026-07-27 00:00:00+00:00,Address,33,3,0.998192,0,0,"CMS, HRSA",NaN,...,NaN,190,447,616,high,[2026-07-27] Severity: high. VT score: 15. Con...,"Group Id: 136273, Group Id: 129288",ms-isac,"CMS, HRSA",No
2,185.93.89.147,2026-07-27 00:00:00+00:00,Address,38,3,0.997918,0,0,"HRSA, NIH, OS",NaN,...,NaN,780,742,526,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 22517998136854987,SystemBC,"HRSA, NIH, OS",Yes
3,45.84.107.172,2026-07-27 00:00:00+00:00,Address,320,3,0.982466,0,0,"CMS, HRSA, OS",NaN,...,pair:russia+gru,590,631,578,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 6755399498003155,Russia,"CMS, HRSA, OS",Yes
4,46.151.182.31,2026-07-27 00:00:00+00:00,Address,12,3,0.999342,0,0,"HRSA, IHS, OS, VA",NaN,...,standalone:command and control,180,433,620,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 4503599631000000,Command and Control,"HRSA, IHS, OS, VA",Yes
5,77.247.126.189,2026-07-27 00:00:00+00:00,Address,94,5,0.994849,0,0,"CMS, FDA, HHS, HRSA, OS",NaN,...,pair:north korea+diamond sleet,280,492,579,high,[2026-07-27] Severity: high. VT score: 11. Con...,"Group Id: 6755399468000794, Group Id: 56294995...",North Korea,"CMS, FDA, HHS, HRSA, OS",Yes
6,93.152.221.13,2026-07-27 00:00:00+00:00,Address,7,3,0.999616,0,0,"HRSA, NIH, OS",NaN,...,NaN,890,824,500,high,[2026-07-27] Severity: high. VT score: 13. Con...,<NA>,NaN,"HRSA, NIH, OS",No


In [25]:
import ipaddress

MIN_HOSTS_PER_SUBNET = 5  # minimum hosts in a /24 before rolling up to CIDR


def _to_ip(value):
    try:
        return ipaddress.ip_address(str(value).strip())
    except ValueError:
        return None


def _subnet24(ip):
    if ip is None:
        return None
    return str(ipaddress.ip_network(f"{ip}/24", strict=False))


def _union_csv(series):
    parts = set()
    for val in series.dropna():
        for part in str(val).split(","):
            part = part.strip()
            if part:
                parts.add(part)
    return ", ".join(sorted(parts)) if parts else None


def _max_severity(series):
    order = {"critical": 2, "high": 1}
    vals = [str(v).strip().lower() for v in series.dropna()]
    if not vals:
        return None
    return max(vals, key=lambda s: order.get(s, 0))


def _has_threat_actor(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return False
    return bool(str(value).strip())


def condense_final_indicators(df, min_hosts=MIN_HOSTS_PER_SUBNET):
    """Roll dense /24 Address clusters into CIDR rows; keep singles and threat-actor IPs."""
    df = df.copy()
    type_col = "Indicator Type" if "Indicator Type" in df.columns else "Type"

    df["_ip"] = df["Indicator"].map(_to_ip)
    df["_subnet24"] = df["_ip"].map(_subnet24)

    ta_col = "Threat Actor" if "Threat Actor" in df.columns else None
    if ta_col:
        df["_has_ta"] = df[ta_col].map(_has_threat_actor)
    else:
        df["_has_ta"] = False

    condensable_mask = (
        df[type_col].eq("Address")
        & df["_ip"].notna()
        & ~df["_has_ta"]
    )

    subnet_counts = df.loc[condensable_mask].groupby("_subnet24").size()
    dense_subnets = set(subnet_counts[subnet_counts >= min_hosts].index)

    dense_mask = condensable_mask & df["_subnet24"].isin(dense_subnets)
    dense_df = df[dense_mask].copy()
    keep_df = df[~dense_mask].copy()

    if dense_df.empty:
        out = df.drop(columns=["_ip", "_subnet24", "_has_ta"], errors="ignore")
        out["_member_ips"] = out["Indicator"].map(lambda x: [str(x)])
        return out

    numeric_max_cols = [
        c for c in [
            "Observation Yearly Count",
            "ThreatConnect Rating",
            "Observation Penalty Multiplier",
            "Botnet Flag",
            "False Positives",
            "Tagging Boost",
            "CAL Score",
            "ThreatConnect Score",
            "PRISM Score",
        ]
        if c in dense_df.columns
    ]

    agg = {c: "max" for c in numeric_max_cols}
    agg.update({
        "Indicator": lambda s: sorted(s.astype(str).tolist()),
        "Last Observed": "max",
        type_col: lambda _: "CIDR",
        "Severity": _max_severity,
        "Partners": _union_csv,
        "OpDiv": _union_csv,
        "Threat Actor": _union_csv,
        "Tagging Boost Reason": _union_csv,
        "Associated Groups": _union_csv,
        "incidents/events": _union_csv,
        "Tags": "first",
        "Explanation": "first",
    })

    if "Reported I&W?" in dense_df.columns:
        agg["Reported I&W?"] = lambda s: "Yes" if (s == "Yes").any() else "No"

    condensed = dense_df.groupby("_subnet24", as_index=False).agg(agg)
    condensed = condensed.rename(columns={"Indicator": "_member_ips"})
    condensed["Indicator"] = condensed["_subnet24"]

    drop_cols = ["_subnet24", "_ip", "_has_ta"]
    condensed = condensed.drop(columns=drop_cols, errors="ignore")
    keep_df = keep_df.drop(columns=drop_cols, errors="ignore")
    keep_df["_member_ips"] = keep_df["Indicator"].map(lambda x: [str(x)])

    out = pd.concat([condensed, keep_df], ignore_index=True, sort=False)

    base_cols = [c for c in df.columns if c not in drop_cols]
    ordered = []
    for col in base_cols:
        ordered.append(col)
        if col == "Indicator" and "_member_ips" in out.columns:
            ordered.append("_member_ips")
    ordered.extend([c for c in out.columns if c not in ordered])
    return out[[c for c in ordered if c in out.columns]]


_before = len(final_indicators)
final_indicators = condense_final_indicators(final_indicators)
print(f"Subnet condensation: {_before} -> {len(final_indicators)} rows (min_hosts={MIN_HOSTS_PER_SUBNET})")

# _member_ips is kept on final_indicators for Excel dropdown export only — not shown here
final_indicators.drop(columns=["_member_ips"], errors="ignore")


Subnet condensation: 7 -> 7 rows (min_hosts=5)


,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,...,Tagging Boost Reason,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,Tags,OpDiv,Reported I&W?
0,176.123.5.126,2026-07-27 00:00:00+00:00,Address,21,5,0.998849,0,0,"HRSA, IHS, OS",NaN,...,cve:cve-2024-50623,360,477,620,high,[2026-07-27] Severity: high. VT score: 15. Con...,"Group Id: 5629499537000808, Group Id: 45035996...",Lace Tempest,"HRSA, IHS, OS",Yes
1,185.220.101.46,2026-07-27 00:00:00+00:00,Address,33,3,0.998192,0,0,"CMS, HRSA",NaN,...,NaN,190,447,616,high,[2026-07-27] Severity: high. VT score: 15. Con...,"Group Id: 136273, Group Id: 129288",ms-isac,"CMS, HRSA",No
2,185.93.89.147,2026-07-27 00:00:00+00:00,Address,38,3,0.997918,0,0,"HRSA, NIH, OS",NaN,...,NaN,780,742,526,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 22517998136854987,SystemBC,"HRSA, NIH, OS",Yes
3,45.84.107.172,2026-07-27 00:00:00+00:00,Address,320,3,0.982466,0,0,"CMS, HRSA, OS",NaN,...,pair:russia+gru,590,631,578,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 6755399498003155,Russia,"CMS, HRSA, OS",Yes
4,46.151.182.31,2026-07-27 00:00:00+00:00,Address,12,3,0.999342,0,0,"HRSA, IHS, OS, VA",NaN,...,standalone:command and control,180,433,620,high,[2026-07-27] Severity: high. VT score: 16. Con...,Group Id: 4503599631000000,Command and Control,"HRSA, IHS, OS, VA",Yes
5,77.247.126.189,2026-07-27 00:00:00+00:00,Address,94,5,0.994849,0,0,"CMS, FDA, HHS, HRSA, OS",NaN,...,pair:north korea+diamond sleet,280,492,579,high,[2026-07-27] Severity: high. VT score: 11. Con...,"Group Id: 6755399468000794, Group Id: 56294995...",North Korea,"CMS, FDA, HHS, HRSA, OS",Yes
6,93.152.221.13,2026-07-27 00:00:00+00:00,Address,7,3,0.999616,0,0,"HRSA, NIH, OS",NaN,...,NaN,890,824,500,high,[2026-07-27] Severity: high. VT score: 13. Con...,<NA>,NaN,"HRSA, NIH, OS",No


In [26]:
from datetime import datetime
import re
from xlsxwriter.utility import xl_rowcol_to_cell

# Build dated output path
today_str = datetime.today().strftime('%Y%m%d')  # e.g. 20260316
output_path = rf"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\ThreatAssessI_W\ThreatAssessI_W_{today_str}.xlsx"

# Excel can't write timezone-aware datetimes; strip tz info before export
_dt_tz_cols = final_indicators.select_dtypes(include=["datetimetz"]).columns
for _c in _dt_tz_cols:
    final_indicators[_c] = final_indicators[_c].dt.tz_convert(None)

iw_col = "Reported I&W?"
if iw_col not in final_indicators.columns:
    raise KeyError(f"Missing required column '{iw_col}' for sheet split.")


def _subnet_range_name(subnet):
    return "MBR_" + re.sub(r"[^A-Za-z0-9]", "_", str(subnet))[:200]


def _prepare_export_df(df):
    export_df = df.copy()
    member_ip_lists = export_df.pop("_member_ips") if "_member_ips" in export_df.columns else pd.Series([None] * len(export_df), index=export_df.index)

    indicator_idx = export_df.columns.get_loc("Indicator") + 1
    export_df.insert(indicator_idx, "Host IP", "")

    member_map = {}
    for idx, ips in member_ip_lists.items():
        if isinstance(ips, list) and len(ips) > 1:
            export_df.at[idx, "Host IP"] = ips[0]
            member_map[export_df.at[idx, "Indicator"]] = ips
        elif isinstance(ips, list) and len(ips) == 1:
            export_df.at[idx, "Host IP"] = ips[0]
        else:
            export_df.at[idx, "Host IP"] = ""

    return export_df, member_map


def _write_subnet_members_sheet(workbook, member_map):
    range_names = {}
    if not member_map:
        return range_names

    ws = workbook.add_worksheet("Subnet_Members")
    ws.hide()

    for col, (subnet, ips) in enumerate(member_map.items()):
        ws.write(0, col, subnet)
        for row, ip in enumerate(ips, start=1):
            ws.write(row, col, ip)

        start = xl_rowcol_to_cell(1, col, row_abs=True, col_abs=True)
        end = xl_rowcol_to_cell(len(ips), col, row_abs=True, col_abs=True)
        name = _subnet_range_name(subnet)
        workbook.define_name(name, f"='Subnet_Members'!{start}:{end}")
        range_names[subnet] = name

    return range_names


def _apply_member_dropdowns(worksheet, sheet_df, member_map, range_names, member_col_name="Host IP"):
    if member_col_name not in sheet_df.columns:
        return

    member_col_idx = sheet_df.columns.get_loc(member_col_name)
    for row_idx, row in enumerate(sheet_df.itertuples(index=False), start=1):
        indicator = getattr(row, "Indicator")
        if indicator not in member_map:
            continue

        ips = member_map[indicator]
        worksheet.data_validation(
            row_idx,
            member_col_idx,
            row_idx,
            member_col_idx,
            {
                "validate": "list",
                "source": f"={range_names[indicator]}",
                "input_title": "Host IP",
                "input_message": f"Select one of {len(ips)} hosts in {indicator}",
            },
        )


export_df, member_map = _prepare_export_df(final_indicators)
final_iw_no = export_df[export_df[iw_col] == "No"].copy()
final_iw_yes = export_df[export_df[iw_col] == "Yes"].copy()

# Write to one workbook with two named sheets
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    final_iw_no.to_excel(writer, index=False, sheet_name="I&W_No")
    final_iw_yes.to_excel(writer, index=False, sheet_name="I&W_Yes")

    workbook = writer.book
    range_names = _write_subnet_members_sheet(workbook, member_map)
    wrap_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})

    # Set defaults first, then tune long text columns for each sheet
    for sheet_name, sheet_df in [("I&W_No", final_iw_no), ("I&W_Yes", final_iw_yes)]:
        worksheet = writer.sheets[sheet_name]
        worksheet.set_column(0, len(export_df.columns) - 1, 18)

        if "Explanation" in export_df.columns:
            _exp_idx = export_df.columns.get_loc("Explanation")
            worksheet.set_column(_exp_idx, _exp_idx, 100, wrap_fmt)

        if "Associated Groups" in export_df.columns:
            _ag_idx = export_df.columns.get_loc("Associated Groups")
            worksheet.set_column(_ag_idx, _ag_idx, 45, wrap_fmt)

        if "Host IP" in export_df.columns:
            _member_idx = export_df.columns.get_loc("Host IP")
            worksheet.set_column(_member_idx, _member_idx, 22)

        _apply_member_dropdowns(worksheet, sheet_df, member_map, range_names)

output_path


'Z:\\HTOC\\Data_Analytics\\Data\\Threat Assessment Scores\\ThreatAssessI_W\\ThreatAssessI_W_20260727.xlsx'